# Logical QFT

Builds one QASM file with Qiskit: QFT-6 on six bb144 blocks with 7 rounds of syndrome extraction after
every logical gate. Logical qubit j of every block forms copy j of QFT-6, so the circuit runs 12 copies.

- **Logical circuit:** `inputs/qasm_files/qft/qft_6.qasm` without its final swaps (a relabelling of
  blocks), as logical `h`, `rz` and `cx` gates.
- **Blocks:** block b is qubits 288b..288b+287 in bb144 order (data 0-143, X ancillas 144-215,
  Z ancillas 216-287).
- **Logical gates:** a logical CNOT a -> b is transversal: `cx` from data qubit i of block a to data
  qubit i of block b. A logical single-qubit gate is that gate on every data qubit of the block. That
  is a stand-in: bb144 has no transversal H or rz.
- **Syndrome extraction:** after each logical gate, on the blocks it acted on: `barrier`, then 7 rounds,
  each followed by a `barrier`. A round is the one in `bb_n144_k12_d12_cz_syndrome.qasm` (X checks,
  then Z checks, with its two barriers), then every ancilla is measured into `syn_b<b>` and reset.
  Ancillas are also reset once at the start.
- **Gate set:** every gate is in the gate set of `qft_n63_transpiled.qasm` (`rz`, `sx`, `cx`); the
  file also contains `measure`, `reset` and `barrier`.

Output: `inputs/qasm_files/logic_qft/qft6_x12_bb144_transversal_qec7.qasm`.

In [17]:
from pathlib import Path
import collections

import qiskit.qasm2
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Clifford, Operator

repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "naive_n_dag").is_dir())  # works from benchmarks/ or the repo root
qasm_dir = repo_root / "inputs" / "qasm_files"
qft6_qasm = qasm_dir / "qft" / "qft_6.qasm"
syndrome_qasm = qasm_dir / "qLDPC" / "bb_n144_k12_d12_cz_syndrome.qasm"
gate_set_qasm = qasm_dir / "qft" / "qft_n63_transpiled.qasm"
output_qasm = qasm_dir / "logic_qft" / "qft6_x12_bb144_transversal_qec7.qasm"
for path in (qft6_qasm, syndrome_qasm, gate_set_qasm):
    assert path.exists(), f"Missing QASM: {path}"

atoms_per_block, data_per_block = 288, 144
syndrome_rounds = 7
non_gates = {"measure", "reset", "barrier"}
native_gates = sorted(set(QuantumCircuit.from_qasm_file(str(gate_set_qasm)).count_ops()) - non_gates)
assert "cx" in native_gates
print("native gate set, from qft_n63_transpiled.qasm:", native_gates)

native gate set, from qft_n63_transpiled.qasm: ['cx', 'rz', 'sx']


In [18]:
# QFT-6 without its final swaps, as logical h / rz / cx gates
qft6 = QuantumCircuit.from_qasm_file(str(qft6_qasm))
body = QuantumCircuit(qft6.num_qubits)
for inst in qft6.data:
    if inst.operation.name != "swap":
        body.append(inst.operation, [qft6.find_bit(q).index for q in inst.qubits])
logical = transpile(body, basis_gates=["h", "rz", "cx"], optimization_level=1, seed_transpiler=0)
n_blocks = logical.num_qubits
print(f"{len(logical.data)} logical gates on {n_blocks} blocks: {dict(logical.count_ops())}")

81 logical gates on 6 blocks: {'rz': 45, 'cx': 30, 'h': 6}


In [19]:
# One syndrome round: the file's gates and two barriers (its resets and measurements dropped), translated
# to the native gate set. The round is Clifford, so the translation must leave its tableau unchanged.
syndrome_file = QuantumCircuit.from_qasm_file(str(syndrome_qasm))
round_gates = QuantumCircuit(syndrome_file.num_qubits)
for inst in syndrome_file.data:
    if inst.operation.name not in ("reset", "measure"):
        round_gates.append(inst.operation, [syndrome_file.find_bit(q).index for q in inst.qubits])
native_round = transpile(round_gates, basis_gates=native_gates, optimization_level=1, seed_transpiler=0)


def without_barriers(circuit: QuantumCircuit) -> QuantumCircuit:
    out = circuit.copy_empty_like()
    for inst in circuit.data:
        if inst.operation.name != "barrier":
            out.append(inst)
    return out


assert Clifford(without_barriers(native_round)) == Clifford(without_barriers(round_gates)), "native round differs"
assert native_round.count_ops()["barrier"] == round_gates.count_ops()["barrier"]
print(f"round: {dict(round_gates.count_ops())} -> native {dict(native_round.count_ops())}")

round: {'cz': 864, 'h': 576, 'barrier': 2} -> native {'rz': 1152, 'cx': 864, 'sx': 576, 'barrier': 2}


In [20]:
# Physical form of each logical gate in the native set. A single-qubit gate is its native decomposition
# on every data qubit of the block (checked equal up to global phase); a CNOT is transversal cx.
def block_atoms(b: int) -> list[int]:
    return list(range(atoms_per_block * b, atoms_per_block * (b + 1)))


def block_data(b: int) -> list[int]:
    return block_atoms(b)[:data_per_block]


def block_ancillas(b: int) -> list[int]:
    return block_atoms(b)[data_per_block:]


def gate_key(op) -> tuple:
    return op.name, tuple(float(p) for p in op.params)


native_1q = {}
for inst in logical.data:
    op = inst.operation
    if op.num_qubits == 1 and gate_key(op) not in native_1q:
        single = QuantumCircuit(1)
        single.append(op, [0])
        native = transpile(single, basis_gates=native_gates, optimization_level=1, seed_transpiler=0)
        assert Operator(native).equiv(Operator(single)), f"{op.name} decomposition differs"
        native_1q[gate_key(op)] = native


def apply_logical_gate(circuit: QuantumCircuit, inst) -> list[int]:
    """Append one logical gate to the physical circuit; return the blocks it acts on."""
    blocks_used = [logical.find_bit(q).index for q in inst.qubits]
    if inst.operation.name == "cx":
        circuit.cx(block_data(blocks_used[0]), block_data(blocks_used[1]))
    else:
        for step in native_1q[gate_key(inst.operation)].data:
            circuit.append(step.operation, [block_data(blocks_used[0])])
    return blocks_used


for (name, params), native in native_1q.items():
    print(f"logical {name}{params if params else ''} -> {[(s.operation.name, [round(float(p), 4) for p in s.operation.params]) for s in native.data]}")

logical h -> [('rz', [1.5708]), ('sx', []), ('rz', [1.5708])]
logical rz(0.7853981633974483,) -> [('rz', [0.7854])]
logical rz(-0.7853981633974483,) -> [('rz', [-0.7854])]
logical rz(0.39269908169872414,) -> [('rz', [0.3927])]
logical rz(-0.39269908169872414,) -> [('rz', [-0.3927])]
logical rz(0.19634954084936207,) -> [('rz', [0.1963])]
logical rz(-0.19634954084936207,) -> [('rz', [-0.1963])]
logical rz(0.09817477042468103,) -> [('rz', [0.0982])]
logical rz(-0.09817477042468103,) -> [('rz', [-0.0982])]
logical rz(0.04908738521234052,) -> [('rz', [0.0491])]
logical rz(-0.04908738521234052,) -> [('rz', [-0.0491])]


In [21]:
# Ancillas reset once; then for every logical gate, on the blocks it acts on:
# gate | barrier | round 1 | barrier | ... | round 7 | barrier
n_ancillas = atoms_per_block - data_per_block
uses = collections.Counter(logical.find_bit(q).index for inst in logical.data for q in inst.qubits)
syndrome_regs = [ClassicalRegister(syndrome_rounds * n_ancillas * uses[b], f"syn_b{b}") for b in range(n_blocks)]
circuit = QuantumCircuit(QuantumRegister(n_blocks * atoms_per_block, "q"), *syndrome_regs)
for b in range(n_blocks):
    circuit.reset(block_ancillas(b))

next_bit = [0] * n_blocks
for inst in logical.data:
    blocks_used = apply_logical_gate(circuit, inst)
    touched = [q for b in blocks_used for q in block_atoms(b)]
    circuit.barrier(touched)
    for _ in range(syndrome_rounds):
        for b in blocks_used:
            circuit.compose(native_round, qubits=block_atoms(b), inplace=True)
            circuit.measure(block_ancillas(b), syndrome_regs[b][next_bit[b]:next_bit[b] + n_ancillas])
            circuit.reset(block_ancillas(b))
            next_bit[b] += n_ancillas
        circuit.barrier(touched)
assert next_bit == [len(reg) for reg in syndrome_regs]

counts = dict(circuit.count_ops())
assert set(counts) <= set(native_gates) | non_gates, f"non-native operations: {set(counts) - set(native_gates) - non_gates}"
block_rounds = syndrome_rounds * sum(uses.values())
print(f"{len(logical.data)} logical gates, {block_rounds} block-rounds of syndrome extraction")
print(f"operations: {counts}")

81 logical gates, 777 block-rounds of syndrome extraction
operations: {'rz': 903312, 'cx': 675648, 'sx': 448416, 'reset': 112752, 'measure': 111888, 'barrier': 2202}


In [22]:
output_qasm.parent.mkdir(parents=True, exist_ok=True)
qiskit.qasm2.dump(circuit, output_qasm)
reread = QuantumCircuit.from_qasm_file(str(output_qasm))
assert dict(reread.count_ops()) == counts and reread.num_qubits == circuit.num_qubits, "written file does not read back the same"
shown = output_qasm.relative_to(repo_root) if output_qasm.is_relative_to(repo_root) else output_qasm
print(f"{output_qasm.stat().st_size / 1e6:.1f} MB, {circuit.num_qubits} qubits, {circuit.num_clbits} syndrome bits -> {shown}")

43.8 MB, 1728 qubits, 111888 syndrome bits -> inputs/qasm_files/logic_qft/qft6_x12_bb144_transversal_qec7.qasm
